# Laporan UTS: Analisis Segmentasi Pelanggan

**Mata Kuliah:** Data Science<br>
**Oleh       :** Kelompok 1<br>
**Offering   :** TI-A

## I. Pendahuluan
Laporan ini bertujuan untuk menganalisis perilaku pelanggan menggunakan teknik data science, termasuk pembersihan data, analisis statistik, reduksi dimensi, dan clustering.<br><br>

## II. Tahapan
Pada bagian ini, fokus pada pembersihan data dan pengelompokan (Exploratory & Segmentation).
1. Data Cleaning (Penanganan Missing Values, Penanganan Data Duplikat, Pengecekan Tipe Data, Pendeteksian Outlier (Pencilan).)
2. Statistics
3. Linear Algebra & Dimensionality Reduction
4. Clustering

## II. Dataset
Dataset yang digunakan dalam analisis ini adalah E-Commerce Behavior Dataset, yang berisi data perilaku pengguna pada sebuah platform e-commerce. Dataset ini mencakup sekitar 8000 pengguna dengan berbagai atribut yang menggambarkan karakteristik pengguna serta aktivitas mereka saat mengunjungi website. Beberapa variabel yang terdapat dalam dataset antara lain usia pengguna (age), waktu yang dihabiskan di website (time_on_site), jumlah halaman yang dilihat (pages_viewed), jumlah barang yang dimasukkan ke keranjang (cart_items), serta informasi mengenai apakah pengguna melihat diskon (discount_seen) dan apakah pengguna merupakan pelanggan yang kembali (returning_user).
Melalui atribut-atribut tersebut, dataset ini memungkinkan analisis terhadap pola perilaku pelanggan dalam berinteraksi dengan platform e-commerce. Data ini dapat digunakan untuk memahami bagaimana karakteristik dan aktivitas pengguna berhubungan dengan keputusan pembelian (purchase). Dengan memanfaatkan teknik analisis data seperti statistik deskriptif, reduksi dimensi menggunakan PCA, serta metode clustering seperti K-Means, dataset ini dapat membantu mengidentifikasi segmen pelanggan yang memiliki perilaku serupa.
Hasil segmentasi ini diharapkan dapat memberikan wawasan mengenai tipe-tipe pelanggan, seperti pengguna yang hanya menjelajah produk, pengguna yang sensitif terhadap diskon, maupun pelanggan loyal yang memiliki kecenderungan melakukan pembelian lebih tinggi.

Dataset:<br>
[E-Commerce Behavior Dataset](https://www.kaggle.com/datasets/asifxzaman/e-commerce-behavior-dataset8000-users)<br>

### 🔵 Data Cleaning

In [16]:
# Library
using CSV, DataFrames, Statistics

In [3]:
# Membaca Dataset
println("Membaca dataset...")
df_raw = CSV.read("../data/df_raw.csv", DataFrame)
println("Data berhasil dimuat!")

Membaca dataset...
Data berhasil dimuat!


In [4]:
# Menghitung Ukuran Data Awal
baris_awal, kolom_awal = size(df_raw)
println("\nJumlah data awal: $baris_awal baris dan $kolom_awal kolom")


Jumlah data awal: 8000 baris dan 14 kolom


In [ ]:
# Penanganan Missing Values: Membersihkan Missing Values dari data mentah (Data Kosong)
df_clean = dropmissing(df_raw)
CSV.write("../data/df_clean.csv", df_clean)
println("Data bersih berhasil disimpan sebagai 'df_clean.csv'")

baris_bersih, _ = size(df_clean)
println("Jumlah data setelah dibersihkan: $baris_bersih baris")
println("Total data yang dihapus: ", baris_awal - baris_bersih, " baris")

Data bersih berhasil disimpan sebagai 'df_clean.csv'
Jumlah data setelah dibersihkan: 6019 baris
Total data yang dihapus: 1981 baris


In [13]:
# Penanganan Data Duplikat: Membersihkan data yang isinya 100% sama persis
baris_sebelum_duplikat = nrow(df_clean)
unique!(df_clean) # Tanda seru (!) artinya langsung mengubah df_clean di memori
baris_setelah_duplikat = nrow(df_clean)
println("Data duplikat yang dihapus: ", baris_sebelum_duplikat - baris_setelah_duplikat, " baris")


Data duplikat yang dihapus: 0 baris


In [ ]:
# Pengecekan Tipe Data: Memastikan kolom berisi value dengan tipe data yang sesuai 
println("\n[Tipe Data Per Kolom]")
for (nama_kolom, tipe_data) in zip(names(df_clean), eltype.(eachcol(df_clean))) # Memastikan Julia membaca kolom angka sebagai Int/Float dan teks sebagai String
    println("- $nama_kolom: $tipe_data")
end


[Tipe Data Per Kolom]
- user_id: Float64
- age: Float64
- gender: String7
- device_type: String7
- time_on_site: Float64
- pages_viewed: Float64
- previous_purchases: Float64
- cart_items: Float64
- discount_seen: Float64
- ad_clicked: Float64
- returning_user: Float64
- avg_session_time: Float64
- bounce_rate: Float64
- purchase: Float64


In [15]:
# Pendeteksi Outlier (Pencilan Logis)
# Pastikan tidak ada data yang "tidak masuk akal"
baris_sebelum_outlier = nrow(df_clean)

filter!(row -> row.age >= 10 && row.age <= 150, df_clean) # Umur harus masuk akal
filter!(row -> row.time_on_site >=0, df_clean) # waktu tidak mungkin negatif

baris_setelah_outlier = nrow(df_clean)
println("\nData outlier yang dihapus: ", baris_sebelum_outlier - baris_setelah_outlier, " baris")


Data outlier yang dihapus: 0 baris


In [19]:
# Simpan ke file csv 
CSV.write("../data/df_clean.csv", df_clean)
println("Data bersih berhasil disimpan sebagai 'df_clean.csv'")

Data bersih berhasil disimpan sebagai 'df_clean.csv'


In [20]:
# Perbandingan df_raw dengan df_clean
# first(df_clean, 5)
describe(df_raw)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,Union
1,user_id,4003.14,1.0,3998.0,8000.0,160,"Union{Missing, Float64}"
2,age,38.5958,18.0,39.0,59.0,160,"Union{Missing, Float64}"
3,gender,,Female,,Male,160,"Union{Missing, String7}"
4,device_type,,Desktop,,Tablet,160,"Union{Missing, String7}"
5,time_on_site,15.7147,1.0,15.74,30.0,160,"Union{Missing, Float64}"
6,pages_viewed,9.97003,1.0,10.0,19.0,160,"Union{Missing, Float64}"
7,previous_purchases,6.96416,0.0,7.0,14.0,160,"Union{Missing, Float64}"
8,cart_items,4.53023,0.0,5.0,9.0,160,"Union{Missing, Float64}"
9,discount_seen,0.502551,0.0,1.0,1.0,160,"Union{Missing, Float64}"


In [21]:
describe(df_clean)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,DataType
1,user_id,4017.59,2.0,4020.0,8000.0,0,Float64
2,age,38.6106,18.0,39.0,59.0,0,Float64
3,gender,,Female,,Male,0,String7
4,device_type,,Desktop,,Tablet,0,String7
5,time_on_site,15.7578,1.0,15.84,30.0,0,Float64
6,pages_viewed,9.98172,1.0,10.0,19.0,0,Float64
7,previous_purchases,6.99169,0.0,7.0,14.0,0,Float64
8,cart_items,4.55358,0.0,5.0,9.0,0,Float64
9,discount_seen,0.502243,0.0,1.0,1.0,0,Float64


### 🔵 Statistics

In [27]:
println("Menggali Insight dari Data Bersih")

# 1. Rangkuman statistik menyeluruh (Nilai Min, Max, Mean, dll)
println("\n1. Statistik Dasar Seluruh Pengunjung ")
tabel_statistik = describe(df_clean)
display(tabel_statistik)

# 2. Analisis spesifik: Perbandingan Perilaku Berdasarkan Perangkat (Device)
insight_device = combine(groupby(df_clean, :device_type),
    nrow => :jumlah_Pengguna,
    :time_on_site => mean => :Rerata_Waktu_Menit,
    :pages_viewed => mean => :Rerata_Halaman_Dilihat,
    :purchase => sum => :Total_Pembelian
)
println("\n2. Perilaku Berdasarkan Tipe Perangkat (Mobile vs Desktop):")
display(insight_device)

# 3. Analisis spesifik: Dampak Diskon Terhadap Pembelian
insight_diskon = combine(groupby(df_clean, :discount_seen),
    :purchase => mean => :Persentase_Beli # Mean dari biner (0/1) akan menghasilkan probabilitas/persentase
)
println("\n3. Apakah melihat diskon meningkatkan peluang beli?:")
display(insight_diskon)

Menggali Insight dari Data Bersih

1. Statistik Dasar Seluruh Pengunjung 


Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Union…,Any,Union…,Any,Int64,DataType
1,user_id,4017.59,2.0,4020.0,8000.0,0,Float64
2,age,38.6106,18.0,39.0,59.0,0,Float64
3,gender,,Female,,Male,0,String7
4,device_type,,Desktop,,Tablet,0,String7
5,time_on_site,15.7578,1.0,15.84,30.0,0,Float64
6,pages_viewed,9.98172,1.0,10.0,19.0,0,Float64
7,previous_purchases,6.99169,0.0,7.0,14.0,0,Float64
8,cart_items,4.55358,0.0,5.0,9.0,0,Float64
9,discount_seen,0.502243,0.0,1.0,1.0,0,Float64



2. Perilaku Berdasarkan Tipe Perangkat (Mobile vs Desktop):


Row,device_type,jumlah_Pengguna,Rerata_Waktu_Menit,Rerata_Halaman_Dilihat,Total_Pembelian
,String7,Int64,Float64,Float64,Float64
1,Mobile,3610,15.961,10.0734,3601.0
2,Desktop,1811,15.3123,9.85146,1809.0
3,Tablet,598,15.8806,9.82274,598.0



3. Apakah melihat diskon meningkatkan peluang beli?:


Row,discount_seen,Persentase_Beli
,Float64,Float64
1,0.0,0.996996
2,1.0,0.999338
